# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-4o-mini"
print("client ready:", MODEL)

client ready: gpt-4o-mini


## 1) Task 1 — 단발 호출 + 토큰/비용 확인

In [2]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
        {"role": "user", "content": "사내 챗봇을 한 문장으로 소개해줘."},
    ],
    temperature=0.7,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

사내 챗봇은 직원들이 신속하게 정보에 접근하고 업무를 효율적으로 수행할 수 있도록 지원하는 AI 기반 도구입니다.
usage: CompletionUsage(completion_tokens=33, prompt_tokens=39, total_tokens=72, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


### 스트리밍 — 토큰이 생성되는 대로 출력

In [3]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "LLM 을 3줄로 설명해줘."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="")
print()

LLM(대형 언어 모델)은 방대한 양의 텍스트 데이터를 기반으로 학습하여 언어 이해와 생성 능력을 가진 인공지능 모델입니다. 자연어 처리(NLP) 작업에서 질문 응답, 번역, 요약 등 다양한 용도로 사용됩니다. 이러한 모델은 텍스트의 의미와 맥락을 파악하여 인간과 비슷한 방식으로 언어를 생성할 수 있습니다.


## 2) Task 2 — 리뷰 100개 긍/부정 분류 (LLM)

실습 1의 산출물 `reviews_clean.parquet` 에서 100건을 샘플링해 분류한다.
- `temperature=0` (분류는 결정적이어야 함)
- **JSON 응답 강제** (`response_format`) — 파싱 안전

In [4]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]                      # 중립 제외 (실습 2 와 동일 기준)
df["label"] = (df["rating"] >= 4).astype(int)   # 정답: 4~5 긍정(1), 1~2 부정(0)
sample = df.sample(100, random_state=42).reset_index(drop=True)
print(sample["label"].value_counts())
sample[["clean_text", "label"]].head()

label
1    52
0    48
Name: count, dtype: int64


,clean_text,label
0,사진과 색이 완전히 달라서 실망했어요,0
1,포장도 꼼꼼하고 품질이 기대 이상이에요,1
2,광고 지금 구매하면 사은품 증정 포장도 꼼꼼하고 품질이 기대 이상이에요,1
3,사이즈도 딱 맞고 마감이 깔끔해요,1
4,배송이 일주일이나 걸렸습니다 별로,0


In [5]:
import json

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    'JSON 으로만 답한다: {"label": 0 또는 1}'
)

def classify(text: str) -> tuple[int, int]:   # (라벨, 사용 토큰) 을 함께 반환
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},   # JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    return int(data["label"]), resp.usage.total_tokens   # 라벨 + 비용계산용 토큰

# 1건 점검 — (라벨, 토큰) 튜플이 찍힌다
label, used = classify(sample.loc[0, "clean_text"])
print("라벨:", label, "/ 토큰:", used)

(0, 82)


In [6]:
preds, tokens = [], 0
for t in sample["clean_text"]:
    label, used = classify(t)
    preds.append(label)
    tokens += used
sample["pred"] = preds
print("총 토큰:", tokens)

총 토큰: 8438


### sklearn(실습 2) 과 비교 — 정확도

In [7]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["label"], sample["pred"])
print(f"LLM 정확도: {acc:.3f}")
print(classification_report(sample["label"], sample["pred"], digits=3))
# TODO: 실습 2 sklearn 정확도와 한 표로 비교 (정확도 / 비용 / 속도 / 일관성)

LLM 정확도: 1.000
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        48
           1      1.000     1.000     1.000        52

    accuracy                          1.000       100
   macro avg      1.000     1.000     1.000       100
weighted avg      1.000     1.000     1.000       100



## 3) Task 3 — 비용 측정 + 1만 건 추산

In [8]:
# gpt-4o-mini 단가 (2026 기준, 변동 가능) — 슬라이드 '비용·운영 한눈에' 참고
# 입력 0.15/M, 출력 0.60/M $. 분류는 출력이 5토큰 내외로 매우 짧아 출력 비용은 무시하고
# total_tokens 를 입력 단가로 보수적으로 추정한다.
PRICE_IN = 0.15 / 1_000_000   # 입력 토큰당 $

avg_tokens = tokens / len(sample)
cost_100   = tokens * PRICE_IN                 # 보수적으로 입력 단가 적용
print(f"평균 토큰/건: {avg_tokens:.1f}")
print(f"100건 비용: ${cost_100:.4f}")
print(f"1만 건 추산: ${cost_100 * 100:.2f}")
# TODO: ML vs LLM — '언제 무엇을 쓸지' 한 줄 결론을 적는다

평균 토큰/건: 84.4
100건 비용: $0.0013
1만 건 추산: $0.13


## 회고 / 산출물
- [ ] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [ ] 한 줄 결론 — *대량·단순 분류는 ___, 유연·복잡 추론은 ___*